# RAG_02: Retrieval

Load the vector DB built by RAG_01 and retrieve similar historical states.
This notebook also exports the core retrieval functions to `rag_retrieval.py`
so that RAG_03 (Optuna weight optimization) can import them directly.

**Input:** `data/rag_embeddings.npz`, `data/rag_metadata.parquet`
**Output:** `rag_retrieval.py` (shared module)

## Export Core Retrieval Module

The cell below writes `rag_retrieval.py` — a standalone module with the 3 core
functions. Both this notebook and RAG_03 import from it.

In [ ]:
%%writefile rag_retrieval.py
"""Core retrieval functions for the RAG conversational state system.

Auto-generated by RAG_02_retrieval.ipynb. Do not edit directly — modify the
notebook cell and re-run.
"""
import numpy as np
import pandas as pd
from pathlib import Path

EMBEDDING_DIM = 1536
WINDOW_NAMES = ['w1', 'w2', 'w3', 'w5', 'w10', 'w_summary']
DEFAULT_WEIGHTS = {name: 1.0 / len(WINDOW_NAMES) for name in WINDOW_NAMES}

# Default file paths (can be overridden in function calls)
_DEFAULT_EMBEDDINGS = Path("data/rag_embeddings.npz")
_DEFAULT_METADATA = Path("data/rag_metadata.parquet")


def load_rag_index(
    embeddings_path=_DEFAULT_EMBEDDINGS,
    metadata_path=_DEFAULT_METADATA,
):
    """Load embeddings and metadata from disk.

    Returns:
        embedding_store: dict of window_name -> np.ndarray of shape (N, 1536)
        metadata_df: pd.DataFrame with N rows aligned to embedding_store
    """
    loaded = np.load(str(embeddings_path))
    embedding_store = {name: loaded[name] for name in WINDOW_NAMES}
    metadata_df = pd.read_parquet(str(metadata_path))

    n = embedding_store[WINDOW_NAMES[0]].shape[0]
    for w in WINDOW_NAMES:
        assert embedding_store[w].shape == (n, EMBEDDING_DIM), \
            f"{w} shape {embedding_store[w].shape} != ({n}, {EMBEDDING_DIM})"
    assert len(metadata_df) == n, \
        f"Metadata rows ({len(metadata_df)}) != embedding rows ({n})"

    return embedding_store, metadata_df


def retrieve_similar_states(
    query_embeddings: dict,
    embedding_store: dict,
    metadata_df: pd.DataFrame,
    weights: dict = None,
    k: int = 5,
    exclude_conversation_id: str = None,
) -> pd.DataFrame:
    """Find top-k most similar states by weighted composite cosine similarity.

    Args:
        query_embeddings: dict of window_name -> 1D numpy array (1536,).
            Should be L2-normalized.
        embedding_store: dict of window_name -> 2D numpy array (N, 1536).
        metadata_df: DataFrame with N rows aligned to embedding_store.
        weights: dict of window_name -> float. Normalized internally.
            Defaults to uniform weights.
        k: number of results to return.
        exclude_conversation_id: if provided, exclude all states from this
            conversation (prevents trivial self-matches).

    Returns:
        DataFrame with k rows from metadata_df plus 'composite_score' column,
        sorted by descending score. Index values are the original positional
        indices into embedding_store / metadata_df.
    """
    if weights is None:
        weights = DEFAULT_WEIGHTS

    # Normalize weights to sum to 1
    total = sum(weights.values())
    weights = {name: w / total for name, w in weights.items()}

    n_states = embedding_store[WINDOW_NAMES[0]].shape[0]
    composite_scores = np.zeros(n_states, dtype='float32')

    for name in WINDOW_NAMES:
        q = np.asarray(query_embeddings[name], dtype='float32')
        q = q / (np.linalg.norm(q) + 1e-8)
        sims = embedding_store[name] @ q  # (n_states,)
        composite_scores += weights[name] * sims

    # Exclude same conversation
    if exclude_conversation_id is not None:
        mask = (metadata_df['conversation_id'].values == exclude_conversation_id)
        composite_scores[mask] = -np.inf

    # Top-k
    top_indices = np.argsort(composite_scores)[::-1][:k]

    result = metadata_df.iloc[top_indices].copy()
    result['composite_score'] = composite_scores[top_indices]
    return result


def retrieve_for_state_index(
    state_idx: int,
    embedding_store: dict,
    metadata_df: pd.DataFrame,
    weights: dict = None,
    k: int = 5,
) -> pd.DataFrame:
    """Retrieve top-k similar states for a state already in the index.

    Uses pre-computed embeddings (no API call needed). Automatically excludes
    the query state's own conversation.
    """
    query_embeddings = {name: embedding_store[name][state_idx] for name in WINDOW_NAMES}
    conv_id = metadata_df.iloc[state_idx]['conversation_id']
    return retrieve_similar_states(
        query_embeddings, embedding_store, metadata_df,
        weights=weights, k=k,
        exclude_conversation_id=conv_id,
    )

## Load Index

In [ ]:
from rag_retrieval import load_rag_index, retrieve_similar_states, retrieve_for_state_index, WINDOW_NAMES

embedding_store, metadata_df = load_rag_index()
print(f"Loaded {len(metadata_df)} states from {metadata_df['conversation_id'].nunique()} conversations")

## Demo: Retrieve Similar States

Pick states from different conversation phases (early, mid, late) and display
the query alongside its top-5 matches.

In [ ]:
import numpy as np

def display_retrieval(query_idx: int, weights=None):
    """Display a query state and its top-5 retrieval results."""
    q = metadata_df.iloc[query_idx]
    print(f"{'='*80}")
    print(f"QUERY: conv={q['conversation_id']}, state={q['state_index']}, "
          f"turns_available={q['n_turns_available']}")
    print(f"  Customer (w1): {q['w1_text'][:200]}")
    print(f"  Agent response: {q['agent_response'][:200]}")
    print(f"  Summary: {q['w_summary_text'][:200]}")
    print()

    results = retrieve_for_state_index(
        query_idx, embedding_store, metadata_df, weights=weights, k=5
    )

    for rank, (idx, row) in enumerate(results.iterrows(), 1):
        print(f"  #{rank} (score={row['composite_score']:.4f}) "
              f"conv={row['conversation_id']}, state={row['state_index']}")
        print(f"     Customer: {row['w1_text'][:150]}")
        print(f"     Agent:    {row['agent_response'][:150]}")
        print()


# Sample states: early, mid, late in the dataset
n = len(metadata_df)
demo_indices = [
    n // 10,       # early
    n // 2,        # mid
    int(n * 0.9),  # late
]

for idx in demo_indices:
    display_retrieval(idx)
    print()

## Per-Window Similarity Breakdown

For a single query, show the individual cosine similarity per window for each
of the top-5 results. This reveals which windows are driving the match and
helps build intuition before running Optuna.

In [ ]:
import pandas as pd

query_idx = len(metadata_df) // 2
results = retrieve_for_state_index(query_idx, embedding_store, metadata_df, k=5)

# Compute per-window similarities for each result
rows = []
for match_idx in results.index:
    per_window = {}
    for name in WINDOW_NAMES:
        q = embedding_store[name][query_idx]
        c = embedding_store[name][match_idx]
        q_norm = q / (np.linalg.norm(q) + 1e-8)
        c_norm = c / (np.linalg.norm(c) + 1e-8)
        per_window[name] = float(q_norm @ c_norm)
    per_window['match_conv'] = metadata_df.iloc[match_idx]['conversation_id']
    per_window['match_state'] = metadata_df.iloc[match_idx]['state_index']
    per_window['composite'] = results.loc[match_idx, 'composite_score']
    rows.append(per_window)

breakdown_df = pd.DataFrame(rows)
breakdown_df = breakdown_df[['match_conv', 'match_state', 'composite'] + WINDOW_NAMES]

q_row = metadata_df.iloc[query_idx]
print(f"Query: conv={q_row['conversation_id']}, state={q_row['state_index']}")
print(f"Customer: {q_row['w1_text'][:150]}")
print()
breakdown_df